In [12]:
!pip install rdkit -qq

In [13]:
!pip install optuna -qq

In [14]:
!pip install catboost

In [15]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from catboost import CatBoostRegressor
from rdkit.Chem import PandasTools
from rdkit import DataStructs
import optuna

In [16]:
df = pd.read_csv('/content/drive/MyDrive/BigSolDB/BigSolDBv2.0.csv')
df = df.dropna()

In [17]:
df_processed = df.copy()

In [18]:
PandasTools.AddMoleculeColumnToFrame(
    df_processed,
    'SMILES_Solute',
    'Mol_Solute')
PandasTools.AddMoleculeColumnToFrame(
    df_processed,
    'SMILES_Solvent',
    'Mol_Solvent')
df_processed.head()

,SMILES_Solute,Temperature_K,Solvent,SMILES_Solvent,Solubility(mole_fraction),Solubility(mol/L),LogS(mol/L),Compound_Name,CAS,PubChem_CID,FDA_Approved,Source,Mol_Solute,Mol_Solvent
0,CCCCCCCCCCCCCCCCCCCC(=O)OCCO,311.25,ethanol,CCO,0.0006,0.010083,-1.996419,Ethylene glycol monoeicosate,26158-80-5,538813.0,No,10.1007/bf00649573,<rdkit.Chem.rdchem.Mol object at 0x7862e156d380>,<rdkit.Chem.rdchem.Mol object at 0x7862e03f9a10>
1,CCCCCCCCCCCCCCCCCCCC(=O)OCCO,314.65,ethanol,CCO,0.0012,0.020100,-1.696799,Ethylene glycol monoeicosate,26158-80-5,538813.0,No,10.1007/bf00649573,<rdkit.Chem.rdchem.Mol object at 0x7862e156ccf0>,<rdkit.Chem.rdchem.Mol object at 0x7862e03f9a80>
2,CCCCCCCCCCCCCCCCCCCC(=O)OCCO,319.15,ethanol,CCO,0.0020,0.033356,-1.476824,Ethylene glycol monoeicosate,26158-80-5,538813.0,No,10.1007/bf00649573,<rdkit.Chem.rdchem.Mol object at 0x7862e156cba0>,<rdkit.Chem.rdchem.Mol object at 0x7862e03f9af0>
3,CCCCCCCCCCCCCCCCCCCC(=O)OCCO,322.15,ethanol,CCO,0.0050,0.083356,-1.079064,Ethylene glycol monoeicosate,26158-80-5,538813.0,No,10.1007/bf00649573,<rdkit.Chem.rdchem.Mol object at 0x7862e156d770>,<rdkit.Chem.rdchem.Mol object at 0x7862e03f9b60>
4,CCCCCCCCCCCCCCCCCCCC(=O)OCCO,324.15,ethanol,CCO,0.0139,0.233286,-0.632111,Ethylene glycol monoeicosate,26158-80-5,538813.0,No,10.1007/bf00649573,<rdkit.Chem.rdchem.Mol object at 0x7862e156cc80>,<rdkit.Chem.rdchem.Mol object at 0x7862e03f9bd0>


In [19]:
from rdkit.Chem import AllChem

def morgan_fp(mol):
  morgan = AllChem.GetMorganGenerator(radius=2, fpSize=512)
  return morgan.GetFingerprint(mol)

In [20]:
df_processed['Morgan_Solute'] = df_processed['Mol_Solute'].apply(morgan_fp)
df_processed['Morgan_Solvent'] = df_processed['Mol_Solvent'].apply(morgan_fp)
df_processed.head()

,SMILES_Solute,Temperature_K,Solvent,SMILES_Solvent,Solubility(mole_fraction),Solubility(mol/L),LogS(mol/L),Compound_Name,CAS,PubChem_CID,FDA_Approved,Source,Mol_Solute,Mol_Solvent,Morgan_Solute,Morgan_Solvent
0,CCCCCCCCCCCCCCCCCCCC(=O)OCCO,311.25,ethanol,CCO,0.0006,0.010083,-1.996419,Ethylene glycol monoeicosate,26158-80-5,538813.0,No,10.1007/bf00649573,<rdkit.Chem.rdchem.Mol object at 0x7862e156d380>,<rdkit.Chem.rdchem.Mol object at 0x7862e03f9a10>,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
1,CCCCCCCCCCCCCCCCCCCC(=O)OCCO,314.65,ethanol,CCO,0.0012,0.020100,-1.696799,Ethylene glycol monoeicosate,26158-80-5,538813.0,No,10.1007/bf00649573,<rdkit.Chem.rdchem.Mol object at 0x7862e156ccf0>,<rdkit.Chem.rdchem.Mol object at 0x7862e03f9a80>,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
2,CCCCCCCCCCCCCCCCCCCC(=O)OCCO,319.15,ethanol,CCO,0.0020,0.033356,-1.476824,Ethylene glycol monoeicosate,26158-80-5,538813.0,No,10.1007/bf00649573,<rdkit.Chem.rdchem.Mol object at 0x7862e156cba0>,<rdkit.Chem.rdchem.Mol object at 0x7862e03f9af0>,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
3,CCCCCCCCCCCCCCCCCCCC(=O)OCCO,322.15,ethanol,CCO,0.0050,0.083356,-1.079064,Ethylene glycol monoeicosate,26158-80-5,538813.0,No,10.1007/bf00649573,<rdkit.Chem.rdchem.Mol object at 0x7862e156d770>,<rdkit.Chem.rdchem.Mol object at 0x7862e03f9b60>,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
4,CCCCCCCCCCCCCCCCCCCC(=O)OCCO,324.15,ethanol,CCO,0.0139,0.233286,-0.632111,Ethylene glycol monoeicosate,26158-80-5,538813.0,No,10.1007/bf00649573,<rdkit.Chem.rdchem.Mol object at 0x7862e156cc80>,<rdkit.Chem.rdchem.Mol object at 0x7862e03f9bd0>,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


In [21]:
X = df_processed[['Temperature_K',
                  'Morgan_Solute',
                  'Morgan_Solvent']]
y = df_processed['LogS(mol/L)']

In [22]:
X['Morgan_Solute'] = X['Morgan_Solute'].tolist()
X['Morgan_Solvent'] = X['Morgan_Solvent'].tolist()

/tmp/ipython-input-1126101153.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X['Morgan_Solute'] = X['Morgan_Solute'].tolist()
/tmp/ipython-input-1126101153.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X['Morgan_Solvent'] = X['Morgan_Solvent'].tolist()


In [23]:
def bitvect_to_array(bitvect):
  arr = np.zeros((1,), dtype=int)
  DataStructs.ConvertToNumpyArray(bitvect, arr)
  return arr

In [24]:
solute_fp = np.vstack(X['Morgan_Solute'].values)
solvent_fp = np.vstack(X['Morgan_Solvent'].values)

In [25]:
X_base = X.drop(['Morgan_Solute', 'Morgan_Solvent'], axis=1)

solute_df = pd.DataFrame(solute_fp, index=X.index).add_prefix('SoluteFP_')
solvent_df = pd.DataFrame(solvent_fp, index=X.index).add_prefix('SolventFP_')

X_final = pd.concat([X_base, solute_df, solvent_df], axis=1)

In [26]:
X_train, X_test, y_train, y_test = train_test_split(
    X_final, y,
    test_size=0.2,
    random_state=42
)

In [27]:
def objective(trial):
  iterations = trial.suggest_int('iterations', 10, 100)
  depth = trial.suggest_int('depth', 3, 10)
  learning_rate = trial.suggest_float('learning_rate', 0.01, 0.1)
  model = CatBoostRegressor(
        iterations=iterations,
        depth=depth,
        learning_rate=learning_rate,
        random_state=42
    )
  model.fit(X_train, y_train)

  y_pred = model.predict(X_test)
  rmse = mean_squared_error(y_test, y_pred)
  r2 = r2_score(y_test, y_pred)

  print(f"Trial {trial.number}: RMSE={rmse:.4f}, R2={r2:.4f}")

  return rmse

In [28]:
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=30)

print(f"Best trial value (RMSE): {study.best_value}")
print(f"Best hyperparameters: {study.best_params}")

[I 2025-10-10 19:59:31,175] A new study created in memory with name: no-name-c533dd14-3068-4542-971e-26c98421cc02


0:	learn: 1.2207620	total: 75.6ms	remaining: 6.88s
1:	learn: 1.2195081	total: 100ms	remaining: 4.5s
2:	learn: 1.2183417	total: 123ms	remaining: 3.66s
3:	learn: 1.2171572	total: 146ms	remaining: 3.21s
4:	learn: 1.2159807	total: 168ms	remaining: 2.93s
5:	learn: 1.2145897	total: 192ms	remaining: 2.75s
6:	learn: 1.2134772	total: 214ms	remaining: 2.6s
7:	learn: 1.2123972	total: 236ms	remaining: 2.48s
8:	learn: 1.2113265	total: 258ms	remaining: 2.38s
9:	learn: 1.2102529	total: 284ms	remaining: 2.33s
10:	learn: 1.2088289	total: 313ms	remaining: 2.31s
11:	learn: 1.2074444	total: 345ms	remaining: 2.3s
12:	learn: 1.2060983	total: 377ms	remaining: 2.29s
13:	learn: 1.2048884	total: 400ms	remaining: 2.23s
14:	learn: 1.2038870	total: 423ms	remaining: 2.17s
15:	learn: 1.2029014	total: 447ms	remaining: 2.12s
16:	learn: 1.2019284	total: 469ms	remaining: 2.07s
17:	learn: 1.2009454	total: 494ms	remaining: 2.03s
18:	learn: 1.1999860	total: 520ms	remaining: 2s
19:	learn: 1.1987786	total: 544ms	remaining: 1

[I 2025-10-10 19:59:39,504] Trial 0 finished with value: 1.298932408472641 and parameters: {'iterations': 92, 'depth': 3, 'learning_rate': 0.014561778648245274}. Best is trial 0 with value: 1.298932408472641.


86:	learn: 1.1461043	total: 2.24s	remaining: 129ms
87:	learn: 1.1455347	total: 2.26s	remaining: 103ms
88:	learn: 1.1449897	total: 2.28s	remaining: 77ms
89:	learn: 1.1443863	total: 2.31s	remaining: 51.3ms
90:	learn: 1.1438382	total: 2.33s	remaining: 25.6ms
91:	learn: 1.1432791	total: 2.35s	remaining: 0us
Trial 0: RMSE=1.2989, R2=0.1191
0:	learn: 1.2158859	total: 207ms	remaining: 17s
1:	learn: 1.2103756	total: 378ms	remaining: 15.3s
2:	learn: 1.2048362	total: 542ms	remaining: 14.4s
3:	learn: 1.1989798	total: 705ms	remaining: 13.9s
4:	learn: 1.1930634	total: 885ms	remaining: 13.8s
5:	learn: 1.1876329	total: 1.04s	remaining: 13.4s
6:	learn: 1.1827906	total: 1.2s	remaining: 13s
7:	learn: 1.1774345	total: 1.38s	remaining: 12.9s
8:	learn: 1.1731623	total: 1.53s	remaining: 12.6s
9:	learn: 1.1679368	total: 1.73s	remaining: 12.7s
10:	learn: 1.1631876	total: 1.93s	remaining: 12.6s
11:	learn: 1.1584070	total: 2.1s	remaining: 12.4s
12:	learn: 1.1542666	total: 2.28s	remaining: 12.3s
13:	learn: 1.149

[I 2025-10-10 20:00:00,629] Trial 1 finished with value: 0.9357490486144466 and parameters: {'iterations': 83, 'depth': 9, 'learning_rate': 0.024222561273035157}. Best is trial 1 with value: 0.9357490486144466.


...

64:	learn: 0.7625349	total: 26s	remaining: 0us
Trial 27: RMSE=0.6009, R2=0.5925
0:	learn: 1.2118774	total: 106ms	remaining: 5.85s
1:	learn: 1.2002539	total: 209ms	remaining: 5.63s
2:	learn: 1.1902251	total: 314ms	remaining: 5.54s
3:	learn: 1.1812838	total: 429ms	remaining: 5.58s
4:	learn: 1.1732205	total: 533ms	remaining: 5.44s
5:	learn: 1.1636826	total: 634ms	remaining: 5.29s
6:	learn: 1.1568827	total: 744ms	remaining: 5.21s
7:	learn: 1.1481765	total: 857ms	remaining: 5.14s
8:	learn: 1.1410558	total: 960ms	remaining: 5.01s
9:	learn: 1.1329389	total: 1.09s	remaining: 5.02s
10:	learn: 1.1261483	total: 1.19s	remaining: 4.86s
11:	learn: 1.1203176	total: 1.29s	remaining: 4.73s
12:	learn: 1.1139823	total: 1.39s	remaining: 4.59s
13:	learn: 1.1085993	total: 1.5s	remaining: 4.5s
14:	learn: 1.1028971	total: 1.59s	remaining: 4.36s
15:	learn: 1.0963432	total: 1.71s	remaining: 4.27s
16:	learn: 1.0915599	total: 1.8s	remaining: 4.14s
17:	learn: 1.0856366	total: 1.9s	remaining: 4.02s
18:	learn: 1.080

[I 2025-10-10 20:08:52,970] Trial 28 finished with value: 0.9129036751930916 and parameters: {'iterations': 56, 'depth': 8, 'learning_rate': 0.048889398065649056}. Best is trial 10 with value: 0.545086205552607.


55:	learn: 0.9513357	total: 5.79s	remaining: 0us
Trial 28: RMSE=0.9129, R2=0.3809
0:	learn: 1.2151749	total: 59.5ms	remaining: 5.35s
1:	learn: 1.2090480	total: 106ms	remaining: 4.7s
2:	learn: 1.2035448	total: 137ms	remaining: 4.01s
3:	learn: 1.1960877	total: 174ms	remaining: 3.79s
4:	learn: 1.1910206	total: 240ms	remaining: 4.13s
5:	learn: 1.1847956	total: 301ms	remaining: 4.26s
6:	learn: 1.1802969	total: 353ms	remaining: 4.24s
7:	learn: 1.1761008	total: 405ms	remaining: 4.2s
8:	learn: 1.1716882	total: 448ms	remaining: 4.08s
9:	learn: 1.1671232	total: 495ms	remaining: 4.01s
10:	learn: 1.1625073	total: 548ms	remaining: 3.98s
11:	learn: 1.1587603	total: 596ms	remaining: 3.92s
12:	learn: 1.1553538	total: 640ms	remaining: 3.84s
13:	learn: 1.1517936	total: 671ms	remaining: 3.69s
14:	learn: 1.1482904	total: 709ms	remaining: 3.59s
15:	learn: 1.1447125	total: 772ms	remaining: 3.62s
16:	learn: 1.1413572	total: 823ms	remaining: 3.58s
17:	learn: 1.1381719	total: 860ms	remaining: 3.49s
18:	learn: 

[I 2025-10-10 20:09:01,403] Trial 29 finished with value: 1.0185837996142573 and parameters: {'iterations': 91, 'depth': 3, 'learning_rate': 0.08009987677244311}. Best is trial 10 with value: 0.545086205552607.


85:	learn: 1.0141927	total: 3.8s	remaining: 221ms
86:	learn: 1.0133616	total: 3.83s	remaining: 176ms
87:	learn: 1.0120607	total: 3.85s	remaining: 131ms
88:	learn: 1.0106901	total: 3.87s	remaining: 87.1ms
89:	learn: 1.0092290	total: 3.9s	remaining: 43.3ms
90:	learn: 1.0080561	total: 3.92s	remaining: 0us
Trial 29: RMSE=1.0186, R2=0.3092
Best trial value (RMSE): 0.545086205552607
Best hyperparameters: {'iterations': 68, 'depth': 10, 'learning_rate': 0.09868370473288476}


In [29]:
catboost_model = CatBoostRegressor(
        iterations=study.best_params['iterations'],
        depth=study.best_params['depth'],
        learning_rate=study.best_params['learning_rate'],
        random_state=42
    )
catboost_model.fit(X_train, y_train)

0:	learn: 1.1949435	total: 353ms	remaining: 23.6s
1:	learn: 1.1697113	total: 701ms	remaining: 23.1s
2:	learn: 1.1455050	total: 1.08s	remaining: 23.4s
3:	learn: 1.1244398	total: 1.43s	remaining: 22.8s
4:	learn: 1.1075744	total: 1.78s	remaining: 22.4s
5:	learn: 1.0893700	total: 2.14s	remaining: 22.2s
6:	learn: 1.0747777	total: 2.49s	remaining: 21.7s
7:	learn: 1.0607428	total: 2.84s	remaining: 21.3s
8:	learn: 1.0473669	total: 3.21s	remaining: 21.1s
9:	learn: 1.0354013	total: 3.56s	remaining: 20.7s
10:	learn: 1.0232883	total: 3.92s	remaining: 20.3s
11:	learn: 1.0131023	total: 4.29s	remaining: 20s
12:	learn: 1.0013075	total: 4.63s	remaining: 19.6s
13:	learn: 0.9892312	total: 4.99s	remaining: 19.2s
14:	learn: 0.9792885	total: 5.42s	remaining: 19.1s
15:	learn: 0.9711041	total: 5.95s	remaining: 19.3s
16:	learn: 0.9630615	total: 6.56s	remaining: 19.7s
17:	learn: 0.9556894	total: 7.18s	remaining: 19.9s
18:	learn: 0.9471939	total: 7.79s	remaining: 20.1s
19:	learn: 0.9401439	total: 8.41s	remaining

In [32]:
from joblib import dump, load
from pathlib import Path

model_dir_path = '/content/drive/MyDrive/CatBoost_solubility'

dump(catboost_model, Path(model_dir_path) / 'catboost_for_solubility_pred.joblib')

['/content/drive/MyDrive/CatBoost_solubility/catboost_for_solubility_pred.joblib']